COMP34812 — Solution 1 (Category A) Demo
Group 34 — AV Track
LightGBM with 695 stylometric features.


# Solution 1 (Category A) — Demo / Inference
## LightGBM Classifier with Comprehensive Stylometric Features

This notebook demonstrates inference with our Category A solution for
Authorship Verification. The model uses ~695 handcrafted stylometric
features including novel syntactic complexity, writing rhythm, and
information-theoretic features, fed into a LightGBM classifier.


In [ ]:
!pip install scikit-learn lightgbm numpy pandas joblib spacy
!python -m spacy download en_core_web_md


In [ ]:
import sys
import numpy as np
import joblib
from pathlib import Path

PROJECT_ROOT = Path('.').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_utils import load_av_data, save_predictions
from src.av_pipeline import AVFeatureExtractor


## 1. Load Trained Model

We load the pre-trained LightGBM classifier, StandardScaler,
and feature name list. The model was trained on 27,643 text pairs
with 695 features per pair.


In [ ]:
scaler = joblib.load('models/av_cat_a_scaler.joblib')
model = joblib.load('models/av_cat_a_lgbm.joblib')
feature_names = joblib.load('models/av_cat_a_feature_names.joblib')
print(f"Model loaded. Features: {len(feature_names)}")


## 2. Load Test Data and Extract Features

We load the dev set (replace with test data path for final submission)
and extract all 695 stylometric features using the AVFeatureExtractor
pipeline. This includes TF-IDF+SVD, spaCy POS/syntactic features,
and all novel feature groups.


In [ ]:
# For demo, use dev data. Replace with test data path for submission.
test_df = load_av_data(split='dev')
print(f"Test data: {len(test_df)} pairs")

extractor = AVFeatureExtractor(use_spacy=True, n_svd_components=100)
# Load pre-fitted TF-IDF components
extractor.tfidf = joblib.load('models/av_cat_a_tfidf.joblib')
extractor.cosine = joblib.load('models/av_cat_a_cosine.joblib')
extractor._fitted = True
extractor._feature_names = feature_names

X_test, _ = extractor.transform(test_df)
X_test_scaled = scaler.transform(X_test)
print(f"Features: {X_test_scaled.shape}")


## 3. Generate Predictions

The LightGBM model predicts binary labels (0 = different author,
1 = same author) for each text pair.


In [ ]:
predictions = model.predict(X_test_scaled)
print(f"Predictions: {len(predictions)}")
print(f"Class distribution: {np.bincount(predictions)}")


## 4. Save Predictions


In [ ]:
save_predictions(predictions, 'predictions/Group_34_A.csv')
print("Saved to predictions/Group_34_A.csv")
